# Workshop 1.2: Indexing and Selecting Data

Welcome to Workshop 1.2! In our previous session, we discovered how pandas bundles data into Series and DataFrames. Now that we have our data neatly arranged in tables, we need precise tools to isolate specific rows or target subsets.

### Precision Filtering in Quantitative Finance

In real market research, you rarely inspect every single line in a dataset. Instead, you filter down to specific trading dates, isolate individual stock tickers, or scan for assets satisfying fundamental criteria. For instance, a quant screener might isolate equities with low valuation multiples that also show strong recent momentum.

Pandas equips us with two complementary indexers: `.loc` for selecting by explicit labels and `.iloc` for selecting by numerical positions. Alongside these indexers, we will explore Boolean masking, which acts like a custom stencil isolating rows that meet our trading criteria.

> **Key Takeaway**: Mastering `.loc`, `.iloc`, and Boolean filtering allows us to slice market tables with speed and surgical precision.

## Topic 1: .loc: Label-Based Selection

The **`.loc[]`** property stands for *location by label*. It allows us to retrieve targeted rows or individual cells using their explicit names.

Think of `.loc[]` like looking up an address in a directory: you specify the exact street and house name rather than counting buildings from the corner. Remember that `.loc[]` uses square brackets rather than parentheses because it is an indexing operator.

Let's create a sample DataFrame with custom row index labels (`"A"`, `"B"`, `"C"`, `"D"`). Let's see:

In [1]:
import pandas as pd

# Create a DataFrame with custom letter index labels:
df = pd.DataFrame({
  "Product": ["Laptop", "Monitor", "Keyboard", "Mouse"],
  "Price": [1200, 300, 50, 25]
}, index=["A", "B", "C", "D"])

print(df)

    Product  Price
A    Laptop   1200
B   Monitor    300
C  Keyboard     50
D     Mouse     25


### Selecting Rows and Cells by Label

We can query our table in multiple ways using `.loc`:
- **Single Row**: Passing a single row name like `df.loc["A"]` extracts that entire row as a Series.
- **Specific Cell**: Passing a row and column label separated by a comma like `df.loc["A", "Price"]` targets that precise value.

Let's see both single row and single cell selections. Let's check:

In [2]:
# We retrieve the entire row labeled 'A':
print("--- Row 'A' ---")
print(df.loc["A"])

# We select the specific value at row 'A' and column 'Price':
print("\n--- Price for Row 'A' ---")
print(df.loc["A", "Price"])

--- Row 'A' ---
Product    Laptop
Price        1200
Name: A, dtype: object

--- Price for Row 'A' ---
1200


### Selecting Multiple Rows and Columns

When we want multiple rows or columns, we pass their names inside a Python list:
- `df.loc[["A", "C"]]`: Selects rows A and C across all columns.
- `df.loc[["A", "C"], ["Product", "Price"]]`: Restricts selection to rows A and C within the specified columns.

Let's inspect multi-label extraction. Let's see:

In [3]:
# Select multiple rows by label:
print("--- Rows 'A' and 'C' ---")
print(df.loc[["A", "C"]])

# Select specific rows AND specific columns:
print("\n--- Rows 'A' & 'C' with Columns 'Product' & 'Price' ---")
print(df.loc[["A", "C"], ["Product", "Price"]])

--- Rows 'A' and 'C' ---
    Product  Price
A    Laptop   1200
C  Keyboard     50

--- Rows 'A' & 'C' with Columns 'Product' & 'Price' ---
    Product  Price
A    Laptop   1200
C  Keyboard     50


> **Key Takeaway**: `.loc[]` targets data using explicit row and column labels, returning a Series for single rows or a DataFrame for multiple entries.

---

## Topic 2: .iloc: Position-Based Selection

Sometimes our tables have arbitrary labels, or we simply want to grab the first row, the last two rows, or a numerical block. For these situations, pandas provides **`.iloc[]`**, which stands for *integer location*.

Unlike `.loc[]`, `.iloc[]` completely ignores custom label names and counts by zero-based integer coordinates:
- Row coordinate 0 represents the first row.
- Column coordinate 0 represents the first column.

Syntax follows the standard coordinate format: `df.iloc[row_position, column_position]`.

Let's test position-based lookups and numerical slices. Let's see:

In [4]:
# We select the first row by its zero-based position:
print("--- First Row (iloc[0]) ---")
print(df.iloc[0])

# We retrieve the cell at row position 0 and column position 1:
print("\n--- Cell at Row 0, Column 1 (iloc[0, 1]) ---")
print(df.iloc[0, 1])

# We slice the first two rows by position:
print("\n--- First Two Rows (iloc[0:2]) ---")
print(df.iloc[0:2])

# We slice both rows and columns simultaneously:
print("\n--- First Two Rows, Column 0 (iloc[0:2, 0:1]) ---")
print(df.iloc[0:2, 0:1])

--- First Row (iloc[0]) ---
Product    Laptop
Price        1200
Name: A, dtype: object

--- Cell at Row 0, Column 1 (iloc[0, 1]) ---
1200

--- First Two Rows (iloc[0:2]) ---
   Product  Price
A   Laptop   1200
B  Monitor    300

--- First Two Rows, Column 0 (iloc[0:2, 0:1]) ---
   Product
A   Laptop
B  Monitor


### Quick Reference: .loc vs .iloc

| Feature | .loc[] | .iloc[] |
| :--- | :--- | :--- |
| **Selection Basis** | **Labels / Names** | **Integer Positions (0, 1, 2...)** |
| **Example** | `df.loc["A", "Price"]` | `df.iloc[0, 1]` |
| **Primary Use** | When row or column names are known | When selecting the Nth row or coordinate ranges |

> **Key Takeaway**: Use `.loc` when referencing data by meaningful names, and `.iloc` when referencing data by numerical coordinates.

## Topic 3: Boolean Filtering

In algorithmic trading, **Boolean filtering** is the mechanism behind stock scanners. It lets us sift through thousands of tickers to locate assets meeting our exact risk and return criteria.

Think of Boolean filtering like placing a stencil over your spreadsheet: rows evaluating to `True` remain visible, while rows evaluating to `False` are filtered out.

Here is how it works under the hood:
- We write a comparison check against a column: `df["Price"] > 100`.
- Pandas evaluates this condition row by row, producing a Boolean Series of `True` and `False` answers.
- When we pass this Boolean Series into the outer selector `df[...]`, pandas keeps only the rows that evaluate to `True`.

Let's see both steps in action. Let's see:

In [5]:
# First, we inspect the boolean series produced by our condition:
print("--- Step 1: The Boolean Series (df['Price'] > 100) ---")
print(df["Price"] > 100)

# Next, we filter our table using the boolean mask:
print("\n--- Step 2: The Filtered DataFrame ---")
print(df[df["Price"] > 100])

--- Step 1: The Boolean Series (df['Price'] > 100) ---
A     True
B     True
C    False
D    False
Name: Price, dtype: bool

--- Step 2: The Filtered DataFrame ---
   Product  Price
A   Laptop   1200
B  Monitor    300


### Combining Multiple Conditions

Trading strategies often check multiple indicators simultaneously. Because pandas evaluates entire columns at once, we use bitwise logical operators:
- **`&`**: Represents logical AND (every condition must evaluate to `True`).
- **`|`**: Represents logical OR (at least one condition must evaluate to `True`).

When combining conditions in pandas, you must enclose each individual check in parentheses, such as `(df["Price"] > 50) & (df["Price"] < 500)`. Forgetting these parentheses is a frequent beginner trap that triggers syntax errors due to Python operator precedence rules.

Let's filter for products falling within a specific price band. Let's check:

In [6]:
# Filter for products with Price > 50 AND Price < 500:
mid_range = df[(df["Price"] > 50) & (df["Price"] < 500)]

print(mid_range)

   Product  Price
B  Monitor    300


> **Key Takeaway**: Boolean filtering isolates rows satisfying logical rules, requiring parentheses around each condition when using `&` and `|`.

---

## Topic 4: Using .isin() for Clean Multi-Value Filters

When filtering for rows matching any item in a target list, writing multiple `|` statements becomes cumbersome. Instead, pandas provides the streamlined **`.isin()`** method.

Passing a list of target items into `.isin()` checks whether each row matches any item in that collection, returning a clean Boolean Series.

Let's filter our table for items matching a target product list. Let's check:

In [7]:
# Filter for products matching items in our list:
target_products = df[df["Product"].isin(["Laptop", "Monitor"])]

print(target_products)

   Product  Price
A   Laptop   1200
B  Monitor    300


> **Key Takeaway**: `.isin(collection)` provides an elegant shortcut for matching rows against a list of candidates.

---

## Topic 5: Querying Data with .query()

As filter conditions grow in complexity, nested brackets like `df[(df[...] > 0) & (df[...] < 10)]` can become visually noisy. To keep code readable, pandas offers the **`.query()`** method.

With `.query()`, we write our filter rule as a clean string expression that directly references column names.

Let's see how readable a `.query()` call can be. Let's see:

In [8]:
# Using .query() with a readable string condition:
expensive_items = df.query("Price > 100")

print(expensive_items)

   Product  Price
A   Laptop   1200
B  Monitor    300


> **Key Takeaway**: The `.query()` method provides an expressive string-based alternative for filtering DataFrames cleanly.

---

## Practice Time

Now it is your turn to practice indexing and filtering DataFrames. Slicing data accurately is a core skill for every quantitative analyst, so work through each challenge carefully.

---

### Challenge 1: Selecting Rows by Label

- Using our inventory DataFrame `df`, use **`.loc`** to select the entire row labeled `"B"`.
- Display the retrieved row on screen.

In [ ]:
# Challenge 1: Write your code below this line


# Expected Output:
# Product  Monitor
# Price     300
# Name: B, dtype: object


### Challenge 2: Slicing Rows by Integer Position

- Using **`.iloc`**, select and print the **last two rows** of `df` (rows C and D).
- Experiment with positive or negative index slices.

In [ ]:
# Challenge 2: Write your code below this line


# Expected Output:
#   Product Price
# C Keyboard   50
# D   Mouse   25


### Challenge 3: Filtering by Numerical Criteria

- Using Boolean filtering, select all products with a price strictly **less than 100** (`Price < 100`).
- Display the filtered DataFrame.

In [ ]:
# Challenge 3: Write your code below this line


# Expected Output:
#   Product Price
# C Keyboard   50
# D   Mouse   25


### Challenge 4: Combining Multi-Column Filters

- First, append a `"Quantity"` column to `df` with values `[5, 10, 20, 15]`.
- Using Boolean filtering with `&`, filter and display all rows where:
  - `Price > 100` and
  - `Quantity > 5`

In [ ]:
# Challenge 4: Write your code below this line


# Expected Output:
#  Product Price Quantity
# B Monitor  300    10


---

## Solutions Section

Terrific job completing these filtering exercises! The ability to isolate specific market slices reliably is essential when evaluating trading strategies.

Let's review the reference implementations together.

### Reference Code

#### Solution for Challenge 1:
```python
print(df.loc["B"])
```

#### Solution for Challenge 2:
```python
print(df.iloc[-2:])
# or print(df.iloc[2:4])
```

#### Solution for Challenge 3:
```python
print(df[df["Price"] < 100])
```

#### Solution for Challenge 4:
```python
df["Quantity"] = [5, 10, 20, 15]
print(df[(df["Price"] > 100) & (df["Quantity"] > 5)])
```

---

### Running the Solutions

Let's run each solution cell to verify our expected outputs:

In [9]:
# Solution for Challenge 1:
print(df.loc["B"])

Product    Monitor
Price          300
Name: B, dtype: object


In [10]:
# Solution for Challenge 2:
print(df.iloc[-2:])

    Product  Price
C  Keyboard     50
D     Mouse     25


In [11]:
# Solution for Challenge 3:
print(df[df["Price"] < 100])

    Product  Price
C  Keyboard     50
D     Mouse     25


In [12]:
# Solution for Challenge 4:
df["Quantity"] = [5, 10, 20, 15]
print(df[(df["Price"] > 100) & (df["Quantity"] > 5)])

   Product  Price  Quantity
B  Monitor    300        10
